# E-Commerce Customer Segmentation using RFM
**Dataset:** `Online Retail.xlsx` (keep this file in the same folder as this notebook)

This notebook cleans transaction data, calculates customer-level RFM features, compares K-Means and DBSCAN, visualizes clusters with PCA and t-SNE, creates simple customer personas, and exports CSV results.

> The provided dataset is synthetic demonstration data, so results are for learning/demo purposes.

In [ ]:
%pip -q install pandas numpy scikit-learn matplotlib seaborn openpyxl

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")

## 1. Load the dataset

In [ ]:
DATA_PATH = "Online Retail.xlsx"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "Online Retail.xlsx not found. Put it in the same folder as this notebook."
    )

raw = pd.read_excel(DATA_PATH)
print("Dataset shape:", raw.shape)
display(raw.head())
print("\\nData types:")
print(raw.dtypes)

## 2. Clean data and calculate RFM

In [ ]:
df = raw.copy()

required = {"InvoiceNo", "InvoiceDate", "CustomerID", "Quantity", "UnitPrice"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {missing}")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["CustomerID"] = df["CustomerID"].astype("string")

df = df.dropna(subset=["InvoiceNo", "InvoiceDate", "CustomerID", "Quantity", "UnitPrice"])
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]  # remove cancellations
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

df["Sales"] = df["Quantity"] * df["UnitPrice"]

# Snapshot is one day after the latest transaction date.
snapshot_date = df["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max().normalize()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Sales", "sum")
).reset_index()

rfm = rfm[(rfm["Frequency"] > 0) & (rfm["Monetary"] > 0)].copy()

print("Rows after cleaning:", len(df))
print("Customers in RFM table:", len(rfm))
display(rfm.head())
display(rfm[["Recency", "Frequency", "Monetary"]].describe().round(2))

## 3. Explore RFM distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["Recency", "Frequency", "Monetary"]):
    sns.histplot(rfm[col], bins=35, ax=ax)
    ax.set_title(f"{col} distribution")
plt.tight_layout()
plt.show()

## 4. Transform and scale features

In [ ]:
features = ["Recency", "Frequency", "Monetary"]
X_log = rfm[features].copy()
X_log["Frequency"] = np.log1p(X_log["Frequency"])
X_log["Monetary"] = np.log1p(X_log["Monetary"])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)
print("Scaled matrix shape:", X_scaled.shape)

## 5. Choose K using Elbow Method and Silhouette Score

In [ ]:
k_values = range(2, min(9, len(rfm)))
inertias, silhouettes = [], []

sample_rng = np.random.default_rng(SEED)
sample_idx = sample_rng.choice(len(X_scaled), size=min(3000, len(X_scaled)), replace=False)
X_sil = X_scaled[sample_idx]

for k in k_values:
    model = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_sil, labels[sample_idx]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_values), inertias, marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")
axes[1].plot(list(k_values), silhouettes, marker="o")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Score")
plt.tight_layout()
plt.show()

best_k = list(k_values)[int(np.argmax(silhouettes))]
print("Selected K based on sampled silhouette:", best_k)

## 6. Fit K-Means and DBSCAN

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=SEED, n_init=10)
rfm["KMeansCluster"] = kmeans.fit_predict(X_scaled)

# Initial DBSCAN settings; tune eps/min_samples if nearly all points are noise or one cluster.
dbscan = DBSCAN(eps=0.8, min_samples=10)
rfm["DBSCANCluster"] = dbscan.fit_predict(X_scaled)

print("K-Means cluster sizes:")
display(rfm["KMeansCluster"].value_counts().sort_index().rename("customers").to_frame())

print("DBSCAN labels (-1 means noise):")
display(rfm["DBSCANCluster"].value_counts().sort_index().rename("customers").to_frame())

def safe_silhouette(X, labels, exclude_noise=False):
    labels = np.asarray(labels)
    mask = labels != -1 if exclude_noise else np.ones(len(labels), dtype=bool)
    valid_labels = labels[mask]
    unique = np.unique(valid_labels)
    if len(unique) < 2 or mask.sum() <= len(unique):
        return np.nan
    X_valid = X[mask]
    if len(X_valid) > 3000:
        idx = np.random.default_rng(SEED).choice(len(X_valid), 3000, replace=False)
        X_valid, valid_labels = X_valid[idx], valid_labels[idx]
    return silhouette_score(X_valid, valid_labels)

print("K-Means silhouette:", round(safe_silhouette(X_scaled, rfm["KMeansCluster"]), 4))
db_score = safe_silhouette(X_scaled, rfm["DBSCANCluster"], exclude_noise=True)
print("DBSCAN silhouette (excluding noise):", db_score)

## 7. PCA visualization

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)
rfm["PCA1"] = X_pca[:, 0]
rfm["PCA2"] = X_pca[:, 1]

print("Variance explained by first two PCs:", round(pca.explained_variance_ratio_.sum(), 3))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=rfm, x="PCA1", y="PCA2", hue="KMeansCluster",
                palette="tab10", s=25, linewidth=0, ax=axes[0])
axes[0].set_title("K-Means clusters (PCA)")
sns.scatterplot(data=rfm, x="PCA1", y="PCA2", hue="DBSCANCluster",
                palette="tab10", s=25, linewidth=0, ax=axes[1])
axes[1].set_title("DBSCAN clusters (PCA)")
plt.tight_layout()
plt.show()

## 8. t-SNE visualization (sample)

In [ ]:
TSNE_SAMPLE = min(1000, len(rfm))
tsne_idx = np.random.default_rng(SEED).choice(len(rfm), size=TSNE_SAMPLE, replace=False)
perplexity = min(30, max(5, (TSNE_SAMPLE - 1) // 3))

tsne = TSNE(
    n_components=2,
    random_state=SEED,
    perplexity=perplexity,
    init="pca",
    learning_rate="auto"
)
X_tsne = tsne.fit_transform(X_scaled[tsne_idx])

tsne_df = rfm.iloc[tsne_idx][["KMeansCluster", "DBSCANCluster"]].copy()
tsne_df["tSNE1"] = X_tsne[:, 0]
tsne_df["tSNE2"] = X_tsne[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=tsne_df, x="tSNE1", y="tSNE2", hue="KMeansCluster",
                palette="tab10", s=25, linewidth=0, ax=axes[0])
axes[0].set_title("K-Means clusters (t-SNE sample)")
sns.scatterplot(data=tsne_df, x="tSNE1", y="tSNE2", hue="DBSCANCluster",
                palette="tab10", s=25, linewidth=0, ax=axes[1])
axes[1].set_title("DBSCAN clusters (t-SNE sample)")
plt.tight_layout()
plt.show()

## 9. Customer personas and suggested actions

In [ ]:
profiles = rfm.groupby("KMeansCluster").agg(
    Customers=("CustomerID", "count"),
    MedianRecency=("Recency", "median"),
    MedianFrequency=("Frequency", "median"),
    MedianMonetary=("Monetary", "median"),
    MeanMonetary=("Monetary", "mean")
).round(2).reset_index()

r_med = rfm["Recency"].median()
f_med = rfm["Frequency"].median()
m_med = rfm["Monetary"].median()

def persona(row):
    recent = row["MedianRecency"] <= r_med
    frequent = row["MedianFrequency"] >= f_med
    high_value = row["MedianMonetary"] >= m_med

    if recent and frequent and high_value:
        return "Champions / VIPs", "Loyalty rewards, early access, referral perks"
    if recent and not frequent:
        return "New or Emerging", "Welcome journey and second-purchase incentive"
    if not recent and frequent:
        return "At-Risk Loyalists", "Win-back message and personalized recommendations"
    if high_value:
        return "High-Spend Occasional", "Premium bundles and exclusive offers"
    if not recent and not frequent:
        return "Hibernating / Low Engagement", "Low-cost reactivation test"
    return "Regular Customers", "Cross-sell complementary products"

persona_values = profiles.apply(persona, axis=1, result_type="expand")
profiles["Persona"] = persona_values[0]
profiles["Suggested action"] = persona_values[1]
display(profiles)

## 10. Plot median RFM profiles

In [ ]:
profile_plot = profiles.set_index("KMeansCluster")[
    ["MedianRecency", "MedianFrequency", "MedianMonetary"]
]
profile_plot.plot(
    kind="bar", figsize=(12, 5), subplots=True,
    layout=(1, 3), legend=False, rot=0
)
plt.suptitle("Median RFM profile by K-Means cluster")
plt.tight_layout()
plt.show()

## 11. Export results

In [ ]:
rfm.to_csv("customer_rfm_clusters.csv", index=False)
profiles.to_csv("customer_segment_personas.csv", index=False)
print("Saved customer_rfm_clusters.csv and customer_segment_personas.csv")